In [1]:
import pandas as pd
from datasets import load_dataset

dataset_dict = load_dataset("piebro/deutsche-bahn-data", data_dir="monthly_processed_data", streaming=True)
df = pd.DataFrame(list(dataset_dict['train'].take(1000000)))
print(df.shape)
print(df.columns.tolist())


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

(1000000, 16)
['station_name', 'xml_station_name', 'eva', 'train_name', 'final_destination_station', 'delay_in_min', 'time', 'is_canceled', 'train_type', 'train_line_ride_id', 'train_line_station_num', 'arrival_planned_time', 'arrival_change_time', 'departure_planned_time', 'departure_change_time', 'id']


In [17]:
df["time"].dtype

dtype('<M8[ns]')

In [19]:
print(df["xml_station_name"].isna().sum())
print((df["xml_station_name"] == "").sum())

0
0


In [2]:
print(df.head())
print(df.describe())

    station_name           xml_station_name       eva train_name  \
0    Dresden Hbf                Dresden Hbf  08010085        S 1   
1            NaN  Hauptbahnhof, Saarbrücken   0836075      STB 1   
2   Nürnberg Hbf               Nürnberg Hbf  08000284     Bus EV   
3    Hamburg Hbf       Hamburg Hbf (S-Bahn)  08098549        S 5   
4  Bielefeld Hbf              Bielefeld Hbf  08000036       RE 6   

      final_destination_station  delay_in_min       time  is_canceled  \
0           Meißen Triebischtal             0 2024-07-01        False   
1  Riegelsberg Süd, Riegelsberg             0 2024-07-01        False   
2                  Nürnberg Hbf             0 2024-07-01        False   
3          Hamburg Elbgaustraße             0 2024-07-01        False   
4                 Bielefeld Hbf            -1 2024-07-01        False   

  train_type    train_line_ride_id  train_line_station_num  \
0          S  -2271920085331621501                      15   
1        STB   -788955727001

In [3]:
df["delay_in_min"].value_counts().head(20)

delay_in_min
 0     367470
 1     197301
 2      98058
 3      65413
 4      45708
 5      33143
 6      25037
 7      19206
-1      16180
 8      14958
 9      12288
 10     10800
 11      8725
 12      7442
 13      6349
 14      5347
 15      5107
 16      4243
 17      3777
-2       3514
Name: count, dtype: int64

In [4]:
print(df["delay_in_min"].skew())

7.245735097961779


In [5]:
df_cancelled = df[df.is_canceled == True]
df_train = df[df.is_canceled == False]
df_train["delay_in_min"] = df_train["delay_in_min"].clip(lower=0)

In the next step we will log-compress the delays, since the data clusteres near 0. (And our loss function squares the error making the model ignore the small values in comparison to rare, large values.)

Since the log(0) is not defined and we have negative delays for trains that came early, we limit the lowest values by 0. 

In [6]:
import numpy as np

df_train["delay_in_min"] = np.log1p(df_train["delay_in_min"]) #claude's solution


Since "df["delay_in_min"] = df["delay_in_min"].apply(lambda x: math.log(x + 1))" loops over every row, Claude suggested the code above.

In [7]:

df_train["delay_in_min"].describe()


count    944682.000000
mean          0.932231
std           0.967961
min           0.000000
25%           0.000000
50%           0.693147
75%           1.609438
max           6.783325
Name: delay_in_min, dtype: float64

In [8]:
df_train["train_line_ride_id"].value_counts()

train_line_ride_id
4464765999393482150     311
-4674302681811850992    305
7474414276230536927     304
-5036776284380011862    304
2686007473625185344     300
                       ... 
-2675144333522528622      1
-5926687760210075747      1
-6141811085571446457      1
355152093825969865        1
-5597218582167690909      1
Name: count, Length: 44564, dtype: int64

In [9]:
df_train[df_train["train_line_ride_id"] == ""]["train_name"].value_counts()

Series([], Name: count, dtype: int64)

Since we have lots of data with no train line ids, we are checking what kind of lines have no ids. Since they are no DB, ICE or IC I will drop them. 

In [10]:
df_train = df_train[df_train.train_line_ride_id != ""]


In [11]:
print(df_train["train_line_ride_id"].value_counts())

train_line_ride_id
4464765999393482150     311
-4674302681811850992    305
7474414276230536927     304
-5036776284380011862    304
2686007473625185344     300
                       ... 
-2675144333522528622      1
-5926687760210075747      1
-6141811085571446457      1
355152093825969865        1
-5597218582167690909      1
Name: count, Length: 44564, dtype: int64


In [12]:
print(len(df_train))

944682


In [13]:
print(df_train["station_name"].nunique())

108


In [14]:
print(df_train["xml_station_name"].nunique())

143


In [15]:
station_avg_delay = df_train.groupby("xml_station_name")["delay_in_min"].mean()

In [16]:
station_pairs = []

for name, group in df.sort_values("train_line_station_num").groupby(["train_line_ride_id", df_train["time"].dt.date]):
    station_list = group["xml_station_name"].tolist()

    station_pairs.extend(list(zip(station_list[:-1], station_list[1:])))
    
unique_pairs = list(set(station_pairs))   
    
print(len(unique_pairs))
print(unique_pairs)


1107
[('Berlin-Spandau', 'Dresden-Neustadt'), ('Aschaffenburg Hbf', 'Hanau Hbf'), ('Münster(Westf)Hbf', 'Gelsenkirchen Hbf'), ('Berlin-Spandau', 'Berlin Zoologischer Garten'), ('Köln Hbf', 'Berlin Hbf'), ('Frankfurt Hbf (tief)', 'Hanau Hbf'), ('Oberhausen Hbf', 'Düsseldorf Flughafen'), ('Mannheim Hbf', 'Mannheim Hbf'), ('Berlin Friedrichstraße (S)', 'Flughafen BER (S-Bahn)'), ('Mannheim Hbf', 'Lüneburg'), ('Berlin-Lichtenberg (S)', 'Berlin Ostbahnhof (S)'), ('Lübeck Hbf', 'Hamburg-Harburg'), ('Berlin-Lichtenberg', 'Flughafen BER'), ('Herford', 'Herford'), ('Neuss Hbf', 'Düsseldorf Flughafen'), ('Berlin Hbf', 'Berlin Gesundbrunnen'), ('Düsseldorf Hbf', 'Hagen Hbf'), ('Würzburg Hbf', 'Erfurt Hbf'), ('Heilbronn Hbf', 'Heidelberg Hbf'), ('Darmstadt Hbf', 'Mainz Hbf'), ('Bamberg', 'Nürnberg Hbf'), ('Lüneburg', 'Uelzen'), ('Frankfurt(Main)Süd', 'Erfurt Hbf'), ('München Hbf Gl.5-10', 'München Ost'), ('Braunschweig Hbf', 'Wolfsburg Hbf'), ('Köln Hbf', 'Köln Hbf'), ('Dresden-Neustadt', 'Hamburg